In [1]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv
import torch
from torch import nn
import pandas as pd
from huggingface_hub import login

logging.basicConfig(
    level=logging.INFO,
    format="%(name)s | %(levelname)s | %(message)s",
)

torch.manual_seed(123)

src_path = Path.cwd().parent / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to sys.path: {src_path}")
load_dotenv()  # reads .env file from the current directory

PATH_DATA = Path.cwd().parent / ".data"
PATH_GPT2_124M_WEIGHTS = PATH_DATA / ".runtime" / "etl_experiment"

Added to sys.path: /home/jtv/code/jtviegas/languagemodels/notebook/src


In [2]:
login(os.getenv("HF_TOKEN"))

httpx | INFO | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
huggingface_hub._login | WARNING | Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
from datasets import load_dataset

dataset = "jtviegas/financial_phrasebank"

train_ds = load_dataset(dataset, split="train")

httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/jtviegas/financial_phrasebank/resolve/main/README.md "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: GET https://huggingface.co/api/datasets/jtviegas/financial_phrasebank "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/jtviegas/financial_phrasebank/resolve/3e83f8bf254692c4f873567632c2e43e515a4408/financial_phrasebank.py "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/jtviegas/financial_phrasebank/jtviegas/financial_phrasebank.py "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/jtviegas/financial_phrasebank/resolve/3e83f8bf254692c4f873567632c2e43e515a4408/README.md "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: GET https://huggingface.co/api/datasets/jtviegas/financial_phrasebank/revision/3e83f8bf254692c4f873567632c2e43e515a4408 "HTTP/1.1 200 OK"
httpx | I

In [4]:
ds = train_ds.train_test_split(test_size=0.2)
train_texts = list(ds["train"]["sentence"])[:100]
test_texts = list(ds["test"]["sentence"])[:100]
train_labels = list(ds["train"]["label"])[:100]
test_labels = list(ds["test"]["label"])[:100]

In [5]:
import tiktoken
from tgedr_lm.classifier.text_dataset import TextDataset

tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = TextDataset(tokenizer=tokenizer, texts=train_texts, labels=train_labels)
val_dataset = TextDataset(tokenizer=tokenizer, texts=test_texts, labels=test_labels)

In [6]:
from tgedr_lm.classifier.gpt2.hyperparam_search import HyperParamSearch
from tgedr_lm.classifier.gpt2.model import GPT2Classifier
from tgedr_lm.configuration import ClassifierBaseConfiguration, TrainingArgs

training_args = TrainingArgs()
hp_search = HyperParamSearch(GPT2Classifier.compute_metrics, train_args=training_args.to_training_arguments())
hyperparameters = hp_search.search(model=GPT2Classifier(ClassifierBaseConfiguration(n_classes=3)), 
                                       train_dataset=train_dataset, val_dataset=val_dataset, 
                                       trials=3)


tgedr_lm.classifier.gpt2.hyperparam_search | INFO | [search|in] (GPT2Classifier(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNormalization()
      (norm2): LayerNormalization()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
 

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F
1,1.117740,1.404746,0.370000,0.277563,0.322222,0.264839
2,0.989210,1.136690,0.260000,0.420139,0.338889,0.148670
3,0.961747,1.164793,0.590000,0.198653,0.327778,0.247379
4,1.207110,1.054739,0.590000,0.198653,0.327778,0.247379
5,0.814015,1.082148,0.590000,0.198653,0.327778,0.247379
6,1.031080,1.129820,0.590000,0.198653,0.327778,0.247379
7,0.936610,0.973457,0.590000,0.198653,0.327778,0.247379
8,0.918130,1.032400,0.590000,0.313292,0.358889,0.319609


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/jtv/code/jtviegas/languagemodels/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/home/jtv/code/jtviegas/languagemodels/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/jtv/code/jtviegas/languagemodels/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-07-12 13:20:14,561] Trial 2 finished with value: 0.62 and parameters: {'learning_rate': 0.0002229897183772444, 'weight_decay': 0.015195408706201964, 'num_train_epochs': 7, 'per_device_train_batch_size': 4, 'warmup_steps': 0.08439977926981096}. Best is trial 2 with value: 0.62.
tgedr_lm.classifier.gpt2.hyperparam_search | INFO | [search|out] => {'learning_rate': 0.0002229897183772444, 'weight_decay': 0.015195408706201964, 'num_train_epochs': 7, 'per_device_train_batch_size': 4, 'warmup_steps': 0.08439977926981096}


In [7]:
from tgedr_lm.configuration import TrainingArgs

training_args = TrainingArgs()
for key in [
    "learning_rate",
    "weight_decay",
    "num_train_epochs",
    "per_device_train_batch_size",
    "warmup_steps",
]:
    if key in hyperparameters:
        training_args.set(key, hyperparameters[key])
        
training_args.set("hub_model_id", "jtviegas/gpt2classifier")
model = GPT2Classifier(ClassifierBaseConfiguration(n_classes=3))

In [8]:
from transformers import Trainer

trainer = Trainer(
        model=model,
        args=training_args.to_training_arguments(),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=model.compute_metrics,
        # Optional: add callbacks for custom behavior
    )

trainer.train()
final_metrics = trainer.evaluate()

accuracy = final_metrics.get("eval_accuracy")
loss = final_metrics.get("eval_loss")
# model.save_pretrained("./final_model")
trainer.push_to_hub(tags="text-classification", commit_message="Training completed!")

/home/jtv/code/jtviegas/languagemodels/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F
1,0.778829,1.888622,0.300000,0.298264,0.345556,0.200231
2,1.137862,0.965178,0.600000,0.200000,0.333333,0.250000
3,1.198560,1.163015,0.590000,0.198653,0.327778,0.247379
4,1.038404,1.798095,0.360000,0.278130,0.324444,0.260362
5,0.808904,1.389381,0.570000,0.248336,0.324444,0.264842
6,0.879857,3.325102,0.580000,0.257683,0.330000,0.268259
7,0.417417,3.000516,0.560000,0.240942,0.318889,0.261430


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/jtv/code/jtviegas/languagemodels/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

httpx | INFO | HTTP Request: POST https://huggingface.co/api/validate-yaml "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: POST https://huggingface.co/api/validate-yaml "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: POST https://huggingface.co/api/models/jtviegas/gpt2classifier/preupload/main "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: POST https://huggingface.co/jtviegas/gpt2classifier.git/info/lfs/objects/batch "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: GET https://huggingface.co/api/models/jtviegas/gpt2classifier/xet-write-token/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

httpx | INFO | HTTP Request: POST https://huggingface.co/api/models/jtviegas/gpt2classifier/commit/main "HTTP/1.1 200 OK"


CommitInfo(commit_url='https://huggingface.co/jtviegas/gpt2classifier/commit/07cdce76cbfed9f7255cd8c39e4296eee20c59d4', commit_message='Training completed!', commit_description='', oid='07cdce76cbfed9f7255cd8c39e4296eee20c59d4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jtviegas/gpt2classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='jtviegas/gpt2classifier'), pr_revision=None, pr_num=None)